In [2]:
from crewai import LLM, Agent, Task, Crew
from crewai.tools import BaseTool

dataset = {
    "CHARACTER": "Pushpa Raj",
    "SOURCE": "Pushpa: The Rise",
    "CATEGORY": "movie character",
    "HUMAN": True,
    "REAL_PERSON": False,
    "FICTIONAL": True,
    "GENDER": "male",
    "Male or female": "male",
    "PROFESSION": "laborer and red sandalwood smuggler",
    "FROM_MOVIE": True,
    "LANGUAGE": "Telugu",
    "FAMOUS": True,
    "HERO": True,
    "VILLAIN": False,
    "BECOMES_RICH": True,
    "CONFIDENT": True,
    "STRONG": True,
    "DANGEROUS": True,
    "HAS_ENEMIES": True,
    "POLICE": False,
    "HAS_FAMILY": True,
    "HAS_ROMANTIC_INTEREST": True,
    "LOVES_RESPECT": True,
    "HAS_FAMOUS_DIALOGUE": True
}



def check_answer(question: str, dataset: dict) -> str:
    q_lower = question.lower().strip().replace('?', '')

    if dataset["CHARACTER"].lower() in q_lower:
        return "BINGO!"

    for key, value in dataset.items():
        if key.lower().replace("_", " ") in q_lower:
            if isinstance(value, bool):
                return "True" if value else "False"
            else:
                return f"True (info: {value})"

    return "False. Inappropriate question"


class CheckTool(BaseTool):
    name: str = "Dataset Checker"
    description: str = "Answers True/False/Inappropriate/BINGO based on dataset."

    def _run(self, query: str) -> str:
        return check_answer(query, dataset)



llm = LLM(model="gemini/gemini-2.0-flash", temperature=0.2)

slave1 = Agent(
    llm=llm,
    verbose=True,
    role="Answer questions",
    goal="Always call Dataset Checker to answer the question with True/False/Inappropriate/BINGO",
    backstory="Street-smart and precise, but ONLY answers from dataset.",
    tools=[CheckTool()]
)

slave2 = Agent(
    llm=llm,
    verbose=True,
    role="Give hints",
    goal="If last 2 answers were False or Inappropriate, give one subtle hint from dataset. If answers were True, say 'You are going in the right direction!'",
    backstory="act as a teasing friend, gives indirect clues."
)


past_answers = [True, True]

while True:
    question = input("\nAsk a question (or guess the character): ")


    work = Task(
        agent=slave1,
        description=f"Answer the question: {question}.",
        expected_output="True, False, Inappropriate, or BINGO!"
    )

    work2 = Task(
        agent=slave2,
        description=f"Based on the last two answers ({past_answers}), give a hint if False/Inappropriate, else encourage.",
        expected_output="A short human-like hint."
    )

    team = Crew(
        agents=[slave1, slave2],
        tasks=[work, work2],
        verbose=False
    )

    result = team.kickoff()

    past_answers[0] = past_answers[1]
    past_answers[1] = work.output.raw

    print(f"Answer: {work.output.raw}")
    print(f"Hint: {work2.output.raw}")

    if "BINGO!" in work.output.raw:
        print("\n🎉 You guessed it right! Game Over.")
        break


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Answer questions                                                                                        │
│                                                                                                                 │
│  Task: Answer the question: What is the gender?.                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Answer questions                                                                                        │
│                                                                                                                 │
│  Thought: I need to use the Dataset Checker to determine if the question "What is the gender?" can be answered  │
│  with True, False, Inappropriate, or BINGO based on the dataset.                                                │
│                                                                                                                 │
│  Using Tool: Dataset Checker                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"query\": \"What is the gender?\"}"                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  True (info: male)                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Answer questions                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  True                                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Give hints                                                                                              │
│                                                                                                                 │
│  Task: Based on the last two answers ([True, True]), give a hint if False/Inappropriate, else encourage.        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (30 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Answer questions                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  BINGO!                                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Give hints                                                                                              │
│                                                                                                                 │
│  Task: Based on the last two answers (['True', 'True']), give a hint if False/Inappropriate, else encourage.    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Give hints                                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  You are going in the right direction!                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Answer: BINGO!
Hint: You are going in the right direction!

🎉 You guessed it right! Game Over.
